# 04 — Visual retrieval baseline

This notebook exercises the **protocol-v0.1 evaluation path**: the visual embedding determines the ranking, while Iconclass labels and hierarchy determine relevance independently. The committed example uses synthetic vectors only to make CI deterministic. It is **not an experimental result**.

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from caypollard.benchmarks.iconclass_retrieval import evaluate_iconclass_retrieval
from caypollard.graphs.iconclass import build_parent_index, child_edges, parse_notations
from caypollard.embeddings.store import EmbeddingTable

## Tiny controlled fixture

The fixture deliberately includes a visual hard negative: item `a` is visually closest to `f`, whose Iconclass label is unresolved. Items `b` and `c` share an exact concept, while other items occupy nearby hierarchy nodes.

In [ ]:
records = [
    {"id": "a", "iconclass": ["25G41"], "split": "test"},
    {"id": "b", "iconclass": ["25G411"], "split": "test"},
    {"id": "c", "iconclass": ["25G411"], "split": "test"},
    {"id": "d", "iconclass": ["25G412"], "split": "test"},
    {"id": "e", "iconclass": ["25G41(+1)"], "split": "test"},
    {"id": "f", "iconclass": ["unresolved"], "split": "test"},
]

vectors = np.asarray([
    [1.00, 0.00],
    [0.00, 1.00],
    [0.00, 0.98],
    [0.10, 0.90],
    [0.80, 0.20],
    [0.99, 0.01],
], dtype=np.float32)

table = EmbeddingTable(
    ids=tuple(row["id"] for row in records),
    vectors=vectors,
    metadata={"fixture": True, "warning": "not a research result"},
)
parents = build_parent_index(
    child_edges(parse_notations(ROOT / "data/samples/iconclass_notations_fixture.txt"))
)

In [ ]:
summary, per_query = evaluate_iconclass_retrieval(
    table, records, parents, query_split="test", candidate_split="test", batch_size=2
)
summary

## Inspect disagreement cases

Per-query output persists visual score, hierarchical relevance, and exact-label overlap. This is the layer used later to identify visual hard negatives and semantic hard positives.

In [ ]:
query_a = next(row for row in per_query if row.query_id == "a")
[(r.item_id, round(r.score, 3), round(r.hierarchical_relevance, 3)) for r in query_a.top_results[:5]]

## Full-corpus command

After extracting real embeddings:

```bash
uv run python scripts/evaluate_iconclass_retrieval.py \
  results/embeddings/dinov2-base.npz \
  data/derived/iconclass-v0.1/manifest.jsonl \
  data/raw/iconclass-core/notations.txt \
  --output-dir results/visual-baseline/dinov2-base
```

The evaluator writes `summary.json`, `per_query.jsonl`, and `per_query.csv`. The summary records corpus/model provenance and stratifies nDCG@10 by fixed label-frequency and hierarchy-depth bins.